In [1]:
import fitz  # PyMuPDF
import re
import pdfplumber

In [2]:
doc = fitz.open(r"C:\Users\Praneeth.kr\Desktop\H_AI\api\uploads\sterling-accuris-pathology-sample-report-unlocked.pdf")
text = "\n".join([page.get_text() for page in doc])

In [3]:
print(text)

Hemoglobin
g/dL
13.0 - 16.5
Colorimetric
14.5
RBC Count
million/cmm
4.5 - 5.5
Electrical impedance
4.79
Hematocrit
%
40 - 49
Calculated
43.3
MCV
fL
83 - 101
Derived
90.3
MCH
pg
27.1 - 32.5
Calculated
30.2
MCHC
g/dL
32.5 - 36.7
Calculated
33.4
RDW CV
%
11.6 - 14
Calculated
13.60
Total WBC and Differential Count
WBC Count
H
/cmm
4000 - 10000
SF Cube cell analysis
10570
Neutrophils
%
40 - 80
Microscopic
73
Lymphocytes
%
20 - 40
Microscopic
19
Eosinophils
%
1 - 6
Microscopic
02
Monocytes
%
2 - 10
Microscopic
06
Basophils
%
0 - 2
Microscopic
00
/cmm
2000 - 6700
7716
/cmm
1100 - 3300
2008
/cmm
00 - 400
211
/cmm
200 - 700
634
/cmm
0 - 100
0
Platelet Count
/cmm
150000 - 410000
Electrical impedance
150000
MPV
H
fL
7.5 - 10.3
Calculated
14.00
Peripheral Smear Examination
RBC Morphology
Normochromic Normocytic
WBC Morphology
WBCs Series Shows Normal Morphology
Platelets Morphology
Platelets are adequate with normal morphology.
Parasites
Malarial parasite is not detected.
Erythrocyte Sedimentation

In [4]:
with pdfplumber.open(r"C:\Users\Praneeth.kr\Desktop\H_AI\api\uploads\sterling-accuris-pathology-sample-report-unlocked.pdf") as pdf:
    all_text = ""
    for page in pdf.pages:
        all_text += page.extract_text() + "\n"
print(all_text)

MC-2202
Scan QR code to check
report authenticity
Passport No : LABORATORY TEST REPORT
Patient Information Sample Information Client/Location Information
Name : Lyubochka Svetka Lab Id : 02232160XXXX Client Name : Sterling Accuris Buddy
Registration on : 20-Feb-2023 09:10
Location :
Sex/Age : Male / 41 Y 01-Feb-1982 Collected at : non SAWPL
Approved on : 20-Feb-2023 11:09 Status : Final
Ref. Id : Collected on : 20-Feb-2023 08:53
Printed On : 28-Feb-2023 10:26
Ref. By : Sample Type : EDTA Blood
Process At : 1. NRL SAWPL Gujarat Ahmedabad Paldi
Complete Blood Count
Test Result Unit Biological Ref. Interval
Hemoglobin 14.5 g/dL 13.0 - 16.5
Colorimetric
RBC Count 4.79 million/cmm 4.5 - 5.5
Electrical impedance
Hematocrit 43.3 % 40 - 49
Calculated
MCV 90.3 fL 83 - 101
Derived
MCH 30.2 pg 27.1 - 32.5
Calculated
MCHC 33.4 g/dL 32.5 - 36.7
Calculated
RDW CV 13.60 % 11.6 - 14
Calculated
Total WBC and Differential Count
WBC Count H10570 /cmm 4000 - 10000
SF Cube cell analysis
Differential Count 

In [13]:
import requests

ollama_url = "http://localhost:11434/api/generate"
ollama_prompt = (
    f"""You are a medical data extraction assistant. Your task is to analyze the following health checkup lab report text and extract ALL lab test parameters and their results in a structured, machine-readable format.

Instructions:
1. Carefully read the entire text below.
2. For every lab parameter or measurement mentioned, extract:
   - "parameter": The exact name as written in the report (avoid standardizing names)
   - "value": The numeric or qualitative result
   - "unit": The unit of measurement (if explicitly mentioned)
   - "reference_range": The normal/reference range provided (verbatim, if available)
   - "status": One of "normal", "high", "low", or "unknown" — based on comparing value to the reference_range
   - "clinical_note": A plain-language comment on the clinical significance, especially for abnormal values

3. Include ALL parameters, even minor or uncommon ones.
4. If a value, unit, or reference range is missing, use the string "unknown".
5. Do NOT include any preamble, summary, or explanation — only the JSON array as output.

Lab report text:
---
{all_text}
---

Genearte a concise and precise response with all important information for medical assistance in such a way that it is fed to the openai for summarization.
"""
)


response = requests.post(
    ollama_url,
    json={
        "model": "gemma3:latest",  # or your preferred local model
        "prompt": ollama_prompt,
        "stream": False
    }
)
summary = response.json()["response"]
print(summary)

Okay, here’s a consolidated medical report summary, designed for optimal input into an AI summarization tool:

**Patient Summary – Urgent Medical Assessment Required**

**Patient:** Lyubochka Svetka
**Date of Collection:** 20-Feb-2023
**Key Abnormalities & Concerns:**

*   **Urinalysis:** Significant abnormalities detected. Urine is pale yellow, with elevated glucose, elevated protein, and casts are absent.
*   **Hemoglobin:** Negative for typical beta thalassemia trait, abnormal peaks detected.
*   **Complete Blood Count:** Hb A 84.4%, Hb A2 2.8%, P2 Peak 5.5%, P3 Peak 5.2%, Foetal Hb 0.3%.
*   **Microscopic:** Elevated glucose, elevated protein,pus cells,red cells, and epithelial cells present.

**Immediate Action Recommended:**  Due to the combination of abnormal urinalysis findings (glucose, protein), the elevated Hb and the presence of other elements, immediate further investigation is warranted to determine the underlying cause. Requires prompt evaluation by a physician.

---

**

In [ ]:
# Optional: Send summarized data to OpenAI for final interpretation
import openai

def create_patient_summary(lab_data):
    """Create a patient-friendly summary using OpenAI"""
    if isinstance(lab_data, str):
        # If Ollama returned text instead of JSON
        summary_text = lab_data
    else:
        # If we have parsed JSON
        summary_text = "Lab Results Summary:\n"
        for item in lab_data:
            param = item.get('parameter', 'Unknown')
            value = item.get('value', 'N/A')
            unit = item.get('unit', '')
            status = item.get('status', 'Unknown')
            significance = item.get('clinical_significance', '')
            
            summary_text += f"- {param}: {value} {unit} ({status})\n"
            if significance:
                summary_text += f"  Clinical Note: {significance}\n"
    
    openai_prompt = f"""
You are a healthcare assistant. Based on the following lab report summary, provide a clear, patient-friendly explanation. 

For each abnormal result:
1. Explain what it means in simple terms
2. Suggest potential causes (if any)
3. Recommend discussing with a doctor if concerning

Lab Summary:
{summary_text}

Please provide a comprehensive but easy-to-understand summary for the patient.
"""
    
    try:
        # Note: You'll need to set your OpenAI API key
        # openai.api_key = "your-api-key-here"
        
        response = openai.ChatCompletion.create(
            model="gpt-4.1",
            messages=[{"role": "user", "content": openai_prompt}],
            max_tokens=512,
            temperature=0.3
        )
        
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error with OpenAI: {e}")
        return None

# Example usage (uncomment to use):
if 'parsed_json' in locals() and parsed_json:
#     patient_summary = create_patient_summary(parsed_json)
#     if patient_summary:
#         print("\n" + "="*50)
#         print("PATIENT-FRIENDLY SUMMARY:")
#         print("="*50)
#         print(patient_summary)


In [ ]:
# Optional: Send summarized data to OpenAI for final interpretation
import openai

def create_patient_summary(lab_data):
    """Create a patient-friendly summary using OpenAI"""
    if isinstance(lab_data, str):
        # If Ollama returned text instead of JSON
        summary_text = lab_data
    else:
        # If we have parsed JSON
        summary_text = "Lab Results Summary:\n"
        for item in lab_data:
            param = item.get('parameter', 'Unknown')
            value = item.get('value', 'N/A')
            unit = item.get('unit', '')
            status = item.get('status', 'Unknown')
            significance = item.get('clinical_significance', '')
            
            summary_text += f"- {param}: {value} {unit} ({status})\n"
            if significance:
                summary_text += f"  Clinical Note: {significance}\n"
    
    openai_prompt = f"""
You are a healthcare assistant. Based on the following lab report summary, provide a clear, patient-friendly explanation. 

For each abnormal result:
1. Explain what it means in simple terms
2. Suggest potential causes (if any)
3. Recommend discussing with a doctor if concerning

Lab Summary:
{summary_text}

Please provide a comprehensive but easy-to-understand summary for the patient.
"""
    
    try:
        # Note: You'll need to set your OpenAI API key
        # openai.api_key = "your-api-key-here"
        
        response = openai.ChatCompletion.create(
            model="gpt-4.1",
            messages=[{"role": "user", "content": openai_prompt}],
            max_tokens=512,
            temperature=0.3
        )
        
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error with OpenAI: {e}")
        return None

# Example usage (uncomment to use):
if 'parsed_json' in locals() and parsed_json:
#     patient_summary = create_patient_summary(parsed_json)
#     if patient_summary:
#         print("\n" + "="*50)
#         print("PATIENT-FRIENDLY SUMMARY:")
#         print("="*50)
#         print(patient_summary)


In [ ]:
# Create a patient-friendly summary using the Ollama processed data
if 'summary' in locals():
    summary_prompt = (
        "Summarize the following health checkup lab report for a patient. "
        "Explain if each value is normal or abnormal and what it might mean in simple terms. "
        "Provide a brief, patient-friendly summary:\n\n"
        f"{summary}\n\n"
        "Provide a comprehensive but easy-to-understand summary for the patient."
    )
    print("OpenAI Prompt prepared:")
    print(summary_prompt[:200] + "...")
else:
    print("No summary available from previous cell. Run Cell 6 first.")
    summary_prompt = None


In [ ]:
# Updated OpenAI API call using the modern format
import openai
from openai import OpenAI

# Initialize OpenAI client (you'll need to set your API key)
# client = OpenAI(api_key="your-api-key-here")

def get_openai_summary(prompt):
    """Send the prompt to OpenAI for patient-friendly summary"""
    try:
        # Uncomment and configure when you have an API key
        # response = client.chat.completions.create(
        #     model="gpt-3.5-turbo",
        #     messages=[{"role": "user", "content": prompt}],
        #     max_tokens=512,
        #     temperature=0.3
        # )
        # return response.choices[0].message.content
        
        # For now, just return a placeholder
        return "OpenAI integration ready. Please add your API key to enable this feature."
    except Exception as e:
        return f"Error with OpenAI: {e}"

# Use the prepared prompt
if 'summary_prompt' in locals() and summary_prompt:
    result = get_openai_summary(summary_prompt)
    print("\nPatient-Friendly Summary:")
    print("=" * 50)
    print(result)
else:
    print("No prompt available. Run the previous cell first.")
